# Doubly Robust Evaluation

## The Doubly Robust Score

Let $q(a,x)$ be a prediction of the reward under action $a$ at context $x$. The doubly robust estimate is

$$
\widehat V_{\mathrm{DR}}(\pi)
=\frac{1}{n}\sum_{i=1}^{n}
\left[
\sum_a\pi(a\mid X_i)\widehat q(a,X_i)
+\frac{\pi(A_i\mid X_i)}{\mu(A_i\mid X_i)}
\{Y_i-\widehat q(A_i,X_i)\}
\right].
$$

The first term averages predicted rewards under the target policy. The second term corrects the prediction using the logged reward. Under the usual one-step causal assumptions, the estimate remains valid when either the behavior probabilities or the reward prediction is correct, subject to regularity conditions.

In [1]:
from pathlib import Path
import io
import sys
import zipfile
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display as _display

def display(value):
    module = value.__class__.__module__
    if module.startswith('pandas') and hasattr(value, '_repr_html_'):
        html = value._repr_html_()
        return _display(HTML('<div style="display:flex;justify-content:center;">' + html + '</div>'))
    return _display(value)
BOOK_ROOT = Path.cwd().parents[1]
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))
from book_support import ensure_dataset
FINANCIAL_ROOT = BOOK_ROOT
import RiskLabAI
from RiskLabAI.causal_factor_analysis import evaluate_policy, select_policy
assert RiskLabAI.__version__ == '3.2.0'
FIGURES = BOOK_ROOT / 'outputs' / 'figures' / '09_policy_evaluation'
TABLES = BOOK_ROOT / 'outputs' / 'tables' / '09_policy_evaluation'
FIGURE_DATA = BOOK_ROOT / 'outputs' / 'figure_data' / '09_policy_evaluation'
DATA_FILE = ensure_dataset(BOOK_ROOT, 'french_49_industries')
for directory in [FIGURES, TABLES, FIGURE_DATA]:
    directory.mkdir(parents=True, exist_ok=True)
PALETTE = {'background': '#111827', 'axes': '#111827', 'text': '#E5E7EB', 'cyan': '#38BDF8', 'amber': '#FBBF24', 'green': '#34D399', 'rose': '#FB7185', 'violet': '#C084FC', 'grid': '#374151'}
plt.rcParams.update({'font.family': 'DejaVu Serif', 'font.size': 11.5, 'figure.facecolor': PALETTE['background'], 'axes.facecolor': PALETTE['axes'], 'axes.edgecolor': PALETTE['text'], 'axes.labelcolor': PALETTE['text'], 'axes.titlecolor': PALETTE['text'], 'xtick.color': PALETTE['text'], 'ytick.color': PALETTE['text'], 'text.color': PALETTE['text'], 'grid.color': PALETTE['grid'], 'grid.linestyle': ':', 'legend.facecolor': PALETTE['axes'], 'legend.edgecolor': PALETTE['grid'], 'savefig.facecolor': PALETTE['background'], 'savefig.transparent': False, 'savefig.dpi': 300})

def rendered_value(value):
    if pd.isna(value):
        return ''
    if isinstance(value, (bool, np.bool_)):
        return 'Pass' if bool(value) else 'Fail'
    if isinstance(value, (int, np.integer)):
        return f'{int(value):,}'
    if isinstance(value, (float, np.floating)):
        value = float(value)
        if value == 0:
            return '0'
        if value.is_integer():
            return f'{int(value):,}'
        if abs(value) < 0.0001 or abs(value) >= 1000000:
            return f'{value:.6g}'
        return f'{value:.6f}'.rstrip('0').rstrip('.')
    return str(value)

def rendered_frame(frame):
    return frame.apply(lambda column: column.map(rendered_value))

def save_table(frame, stem, caption, label, fit=False):
    shown = rendered_frame(frame)
    shown.to_csv(TABLES / f'{stem}.csv', index=False)
    latex = shown.to_latex(index=False, escape=True, caption=caption, label=label, position='htbp')
    latex = latex.replace('\\begin{table}[htbp]', '\\begin{table}[htbp]\n\\centering', 1)
    if fit:
        latex = latex.replace('\\begin{tabular}', '\\begin{adjustbox}{max width=\\textwidth}\\begin{tabular}', 1).replace('\\end{tabular}', '\\end{tabular}\\end{adjustbox}', 1)
    (TABLES / f'{stem}.tex').write_text(latex, encoding='utf-8')
    display(shown)
    return shown

def finish_axis(axis, grid=True):
    axis.spines['top'].set_visible(False)
    axis.spines['right'].set_visible(False)
    if grid:
        axis.grid(True, alpha=0.65)
    axis.set_axisbelow(True)

def save_figure(figure, stem):
    figure.savefig(FIGURES / f'{stem}.png', dpi=300, bbox_inches='tight', facecolor=PALETTE['background'], transparent=False)
    figure.savefig(FIGURES / f'{stem}.pdf', bbox_inches='tight', facecolor=PALETTE['background'], transparent=False)
    display(figure)
    plt.close(figure)

def read_public_returns():
    with zipfile.ZipFile(DATA_FILE) as archive:
        lines = archive.read(archive.namelist()[0]).decode('latin1').splitlines()
    header = next((index for index, line in enumerate(lines) if line.startswith(',Agric')))
    ending = next((index for index in range(header + 1, len(lines)) if not lines[index].strip()))
    raw = pd.read_csv(io.StringIO('\n'.join(lines[header:ending])))
    raw = raw.rename(columns={raw.columns[0]: 'Date'})
    raw['Date'] = pd.to_datetime(raw['Date'].astype(str), format='%Y%m%d')
    return raw.set_index('Date').replace([-99.99, -999.0], np.nan) / 100

def logistic(values):
    clipped = np.clip(values, -35, 35)
    return 1 / (1 + np.exp(-clipped))

def build_public_frame(floor=0.1, random_state=9101):
    returns = read_public_returns()
    defensive = ['Food', 'Smoke', 'Hlth', 'Util']
    cyclical = ['BldMt', 'Steel', 'Mach', 'Autos']
    baskets = pd.DataFrame({'Defensive return': returns[defensive].mean(axis=1), 'Cyclical return': returns[cyclical].mean(axis=1)})
    spread = baskets['Cyclical return'] - baskets['Defensive return']
    market = baskets.mean(axis=1)
    context = pd.DataFrame({'Past mean spread': spread.rolling(20).mean().shift(1), 'Past volatility': market.rolling(20).std(ddof=1).shift(1), 'Past one-day spread': spread.shift(1)})
    reference = context.loc['2018-01-01':'2018-12-31'].dropna()
    center = reference.mean()
    scale = reference.std(ddof=1)
    standardized = (context - center) / scale
    frame = pd.concat([baskets, context, standardized.add_suffix(' z')], axis=1).loc['2019-01-01':'2023-12-31'].dropna().copy()
    score = 0.7 * frame['Past mean spread z'] - 0.4 * frame['Past volatility z'] + 0.2 * frame['Past one-day spread z']
    probability_one = floor + (1 - 2 * floor) * logistic(score)
    uniforms = np.random.default_rng(random_state).random(len(frame))
    actions = (uniforms < probability_one.to_numpy()).astype(int)
    potential = frame[['Defensive return', 'Cyclical return']].to_numpy()
    frame['Behavior probability action 1'] = probability_one
    frame['Behavior probability action 0'] = 1 - probability_one
    frame['Action'] = actions
    frame['Observed reward'] = potential[np.arange(len(frame)), actions]
    frame['Uniform draw'] = uniforms
    return frame

def behavior_matrix(frame):
    return frame[['Behavior probability action 0', 'Behavior probability action 1']].to_numpy()

def potential_matrix(frame):
    return frame[['Defensive return', 'Cyclical return']].to_numpy()

def candidate_policies(frame):
    spread = frame['Past mean spread z'].to_numpy()
    volatility = frame['Past volatility z'].to_numpy()
    probabilities_one = [np.zeros(len(frame)), np.ones(len(frame)), (spread > 0).astype(float), logistic(1.5 * spread), logistic(1.2 * spread - 0.6 * volatility)]
    candidates = np.stack([np.column_stack([1 - values, values]) for values in probabilities_one])
    names = ['Always defensive', 'Always cyclical', 'Positive spread threshold', 'Smooth momentum', 'Volatility aware rule']
    return (names, candidates)

def full_information_value(frame, policy, indices=None):
    rows = np.arange(len(frame)) if indices is None else np.asarray(indices)
    return float(np.mean(np.sum(policy[rows] * potential_matrix(frame)[rows], axis=1)))

def design_matrix(frame):
    return np.column_stack([np.ones(len(frame)), frame['Past mean spread z'].to_numpy(), frame['Past volatility z'].to_numpy(), frame['Past one-day spread z'].to_numpy()])

def fitted_coefficients(frame, training_rows):
    features = design_matrix(frame)
    actions = frame['Action'].to_numpy()
    rewards = frame['Observed reward'].to_numpy()
    coefficients = []
    for action in [0, 1]:
        rows = np.asarray(training_rows)[actions[np.asarray(training_rows)] == action]
        coefficients.append(np.linalg.lstsq(features[rows], rewards[rows], rcond=None)[0])
    return np.asarray(coefficients)

def chronological_predictions(frame):
    boundaries = [200, 370, 540, 710, 880, len(frame)]
    features = design_matrix(frame)
    predictions = np.zeros((len(frame), 2))
    folds = []
    for start, end in zip(boundaries[:-1], boundaries[1:]):
        coefficients = fitted_coefficients(frame, np.arange(start))
        predictions[start:end] = features[start:end] @ coefficients.T
        folds.append({'Training first date': str(frame.index[0].date()), 'Training last date': str(frame.index[start - 1].date()), 'Prediction first date': str(frame.index[start].date()), 'Prediction last date': str(frame.index[end - 1].date()), 'Training rows': start, 'Prediction rows': end - start, 'Training precedes prediction': bool(frame.index[start - 1] < frame.index[start])})
    return (predictions, pd.DataFrame(folds))

def capture_failure(name, calculation):
    try:
        calculation()
        return {'Check': name, 'Rejected': False, 'Explanation': 'Calculation continued'}
    except ValueError:
        return {'Check': name, 'Rejected': True, 'Explanation': 'Stopped as required'}
frame = build_public_frame()
names, candidates = candidate_policies(frame)
behavior = behavior_matrix(frame)
potential = potential_matrix(frame)
actions = frame['Action'].to_numpy()
rewards = frame['Observed reward'].to_numpy()
predictions, fold_plan = chronological_predictions(frame)
exact_actions = np.array([0, 0, 0, 1])
exact_rewards = np.array([2.0, 2.0, 2.0, 6.0])
correct_behavior = np.tile([0.75, 0.25], (4, 1))
wrong_behavior = np.tile([0.5, 0.5], (4, 1))
target = np.tile([0.4, 0.6], (4, 1))
correct_outcome = np.tile([2.0, 6.0], (4, 1))
wrong_outcome = np.zeros((4, 2))
exact_cases = []
exact_score_rows = []
for name, supplied_behavior, supplied_outcome in [('Behavior correct, outcome wrong', correct_behavior, wrong_outcome), ('Behavior wrong, outcome correct', wrong_behavior, correct_outcome), ('Both correct', correct_behavior, correct_outcome), ('Both wrong', wrong_behavior, wrong_outcome)]:
    result = evaluate_policy(exact_actions, exact_rewards, supplied_behavior, target, supplied_outcome)
    exact_cases.append({'Case': name, 'IPW value': result['ipw_value'], 'DR value': result['dr_value'], 'Known value': 4.4, 'DR absolute error': abs(result['dr_value'] - 4.4)})
    for index in range(4):
        exact_score_rows.append({'Case': name, 'Row': index + 1, 'Action': exact_actions[index], 'Reward': exact_rewards[index], 'Weight': result['importance_weights'][index], 'IPW score': result['ipw_scores'][index], 'DR score': result['dr_scores'][index]})
exact_cases = pd.DataFrame(exact_cases)
exact_scores = pd.DataFrame(exact_score_rows)
save_table(exact_cases, 'dr_exact_cases', 'Exact four-row double robustness check.', 'tab:cim9-dr-exact-cases', fit=True)
save_table(exact_scores, 'dr_exact_scores', 'Row contributions in the exact double robustness check.', 'tab:cim9-dr-exact-scores', fit=True)
fold_plan.insert(0, 'Fold', np.arange(1, len(fold_plan) + 1))
save_table(fold_plan, 'dr_fold_plan', 'Chronological reward-prediction folds.', 'tab:cim9-dr-fold-plan', fit=True)
oof_checks = pd.DataFrame([{'Requirement': 'Every prediction fold follows its training dates', 'Status': bool(fold_plan['Training precedes prediction'].all())}, {'Requirement': 'First policy-selection prediction starts after burn-in', 'Status': bool(fold_plan.iloc[0]['Training rows'] == 200)}, {'Requirement': 'Held-out predictions use only earlier dates', 'Status': bool(fold_plan.iloc[-1]['Training rows'] == 880)}, {'Requirement': 'Prediction array matches rows and actions', 'Status': bool(predictions.shape == behavior.shape)}])
save_table(oof_checks, 'dr_oof_checks', 'Information-boundary checks for reward predictions.', 'tab:cim9-dr-oof-checks', fit=True)
prediction_rows = []
evaluation_rows = np.arange(200, len(frame))
for action, label in [(0, 'Defensive'), (1, 'Cyclical')]:
    actual = potential[evaluation_rows, action]
    predicted = predictions[evaluation_rows, action]
    prediction_rows.append({'Action': action, 'Basket': label, 'Rows': len(actual), 'Mean error': np.mean(predicted - actual), 'Mean absolute error': np.mean(np.abs(predicted - actual)), 'Root mean squared error': np.sqrt(np.mean((predicted - actual) ** 2)), 'Correlation': np.corrcoef(predicted, actual)[0, 1]})
prediction_quality = pd.DataFrame(prediction_rows)
save_table(prediction_quality, 'dr_prediction_quality', 'Quality of chronological reward predictions against both public basket returns.', 'tab:cim9-dr-prediction-quality', fit=True)
sample_rows = np.linspace(200, 879, 8, dtype=int)
oof_sample = pd.DataFrame({'Date': frame.index[sample_rows].date.astype(str), 'Observed action': actions[sample_rows], 'Observed reward': rewards[sample_rows], 'Predicted defensive reward': predictions[sample_rows, 0], 'Predicted cyclical reward': predictions[sample_rows, 1]})
save_table(oof_sample, 'dr_oof_sample_rows', 'Selected chronological reward predictions from the policy-selection period.', 'tab:cim9-dr-oof-sample-rows', fit=True)
exact_cases.to_csv(FIGURE_DATA / 'dr_exact_cases.csv', index=False)
figure, axis = plt.subplots(figsize=(9.0, 5.4), constrained_layout=True)
positions = np.arange(len(exact_cases))
axis.bar(positions - 0.18, exact_cases['IPW value'], 0.36, color=PALETTE['cyan'], label='IPW')
axis.bar(positions + 0.18, exact_cases['DR value'], 0.36, color=PALETTE['amber'], label='Doubly robust')
axis.axhline(4.4, color=PALETTE['green'], linewidth=2, linestyle='--', label='Known value')
axis.set_xticks(positions, ['Behavior\ncorrect', 'Outcome\ncorrect', 'Both\ncorrect', 'Both\nwrong'])
axis.set_ylabel('Policy value')
axis.set_title('Exact Double Robustness Check')
axis.legend()
finish_axis(axis)
save_figure(figure, 'dr_exact_cases')
for action, stem, label, color in [(0, 'dr_oof_action0', 'Defensive', PALETTE['cyan']), (1, 'dr_oof_action1', 'Cyclical', PALETTE['amber'])]:
    plot_data = pd.DataFrame({'Actual reward': potential[evaluation_rows, action], 'Predicted reward': predictions[evaluation_rows, action]})
    plot_data.to_csv(FIGURE_DATA / f'{stem}.csv', index=False)
    figure, axis = plt.subplots(figsize=(6.8, 5.8), constrained_layout=True)
    axis.scatter(plot_data['Actual reward'], plot_data['Predicted reward'], color=color, s=15, alpha=0.45)
    limits = [min(plot_data.min()), max(plot_data.max())]
    axis.plot(limits, limits, color=PALETTE['green'], linewidth=1.5, linestyle='--', label='Equal values')
    axis.set_xlabel('Public basket return')
    axis.set_ylabel('Chronological prediction')
    axis.set_title(f'{label} Reward Predictions')
    axis.legend()
    finish_axis(axis)
    save_figure(figure, stem)


,Case,IPW value,DR value,Known value,DR absolute error
0,"Behavior correct, outcome wrong",4.4,4.4,4.4,8.88178e-16
1,"Behavior wrong, outcome correct",3,4.4,4.4,8.88178e-16
2,Both correct,4.4,4.4,4.4,8.88178e-16
3,Both wrong,3,3,4.4,1.4


,Case,Row,Action,Reward,Weight,IPW score,DR score
0,"Behavior correct, outcome wrong",1,0,2,0.533333,1.066667,1.066667
1,"Behavior correct, outcome wrong",2,0,2,0.533333,1.066667,1.066667
2,"Behavior correct, outcome wrong",3,0,2,0.533333,1.066667,1.066667
3,"Behavior correct, outcome wrong",4,1,6,2.4,14.4,14.4
4,"Behavior wrong, outcome correct",1,0,2,0.8,1.6,4.4
5,"Behavior wrong, outcome correct",2,0,2,0.8,1.6,4.4
6,"Behavior wrong, outcome correct",3,0,2,0.8,1.6,4.4
7,"Behavior wrong, outcome correct",4,1,6,1.2,7.2,4.4
8,Both correct,1,0,2,0.533333,1.066667,4.4
9,Both correct,2,0,2,0.533333,1.066667,4.4


,Fold,Training first date,Training last date,Prediction first date,Prediction last date,Training rows,Prediction rows,Training precedes prediction
0,1,2019-01-02,2019-10-16,2019-10-17,2020-06-19,200,170,Pass
1,2,2019-01-02,2020-06-19,2020-06-22,2021-02-23,370,170,Pass
2,3,2019-01-02,2021-02-23,2021-02-24,2021-10-25,540,170,Pass
3,4,2019-01-02,2021-10-25,2021-10-26,2022-06-29,710,170,Pass
4,5,2019-01-02,2022-06-29,2022-06-30,2023-12-29,880,378,Pass


,Requirement,Status
0,Every prediction fold follows its training dates,Pass
1,First policy-selection prediction starts after...,Pass
2,Held-out predictions use only earlier dates,Pass
3,Prediction array matches rows and actions,Pass


,Action,Basket,Rows,Mean error,Mean absolute error,Root mean squared error,Correlation
0,0,Defensive,"1,058",0.000305,0.008046,0.012588,0.070791
1,1,Cyclical,"1,058",0.000482,0.015088,0.022091,0.003264


,Date,Observed action,Observed reward,Predicted defensive reward,Predicted cyclical reward
0,2019-10-17,0,0.008325,-0.000839,0.004006
1,2020-03-09,1,-0.108625,0.001265,0.01946
2,2020-07-27,0,0.00555,0.001449,-0.001824
3,2020-12-11,0,-0.0034,0.002664,0.001458
4,2021-05-04,0,-0.003525,-0.0017,0.001148
5,2021-09-21,0,-0.0011,-0.000621,0.001984
6,2022-02-08,1,0.023075,-0.002093,-0.001426
7,2022-06-29,0,-0.0021,0.000316,-0.003689


<Figure size 900x540 with 1 Axes>

<Figure size 680x580 with 1 Axes>

<Figure size 680x580 with 1 Axes>

## Evaluating policy values from chronological predictions

The reward predictions used in the policy-selection period are formed in blocks. For each block, the model is fitted only on earlier logged rewards. The final period uses one model fitted on all earlier dates. This ordering prevents a date's outcome from helping predict itself or choose the final policy.

Because both basket returns are public, the full-information value can also be calculated as

$$
V_{\mathrm{full}}(\pi)
=\frac{1}{n}\sum_{i=1}^{n}\sum_{a\in\{0,1\}}
\pi(a\mid X_i)Y_i(a).
$$

Here, $Y_i(a)$ is the observed return of basket $a$ on date $i$. This benchmark checks the logged-data estimates in the illustration; it would normally be unavailable when only the chosen action's reward is observed.

In [2]:
heldout = np.arange(880, len(frame))
comparison_rows = []
for name, policy in zip(names, candidates):
    result = evaluate_policy(actions[heldout], rewards[heldout], behavior[heldout], policy[heldout], predictions[heldout])
    truth = full_information_value(frame, policy, heldout)
    comparison_rows.append({"Policy": name, "Full-information value": truth, "IPW value": result["ipw_value"], "DR value": result["dr_value"], "IPW absolute error": abs(result["ipw_value"] - truth), "DR absolute error": abs(result["dr_value"] - truth), "Effective sample size": result["effective_sample_size"]})
comparison = pd.DataFrame(comparison_rows)
save_table(comparison, "dr_policy_comparison", "Held-out full-information, inverse probability, and doubly robust values.", "tab:cim9-dr-policy-comparison", fit=True)

smooth = candidates[3]
smooth_result = evaluate_policy(actions[heldout], rewards[heldout], behavior[heldout], smooth[heldout], predictions[heldout])
score_summary = pd.DataFrame([
    {"Score": "IPW", "Mean": smooth_result["ipw_scores"].mean(), "Standard deviation": smooth_result["ipw_scores"].std(ddof=1), "Minimum": smooth_result["ipw_scores"].min(), "Median": np.median(smooth_result["ipw_scores"]), "Maximum": smooth_result["ipw_scores"].max()},
    {"Score": "Doubly robust", "Mean": smooth_result["dr_scores"].mean(), "Standard deviation": smooth_result["dr_scores"].std(ddof=1), "Minimum": smooth_result["dr_scores"].min(), "Median": np.median(smooth_result["dr_scores"]), "Maximum": smooth_result["dr_scores"].max()},
])
save_table(score_summary, "dr_score_summary", "Held-out score distributions for the smooth momentum policy.", "tab:cim9-dr-score-summary", fit=True)

oracle = potential.copy()
wrong_probability = np.full_like(behavior, 0.5)
zero_prediction = np.zeros_like(predictions)
diagnostic_rows = np.arange(200, len(frame))
nuisance_rows = []
for name, supplied_behavior, supplied_predictions in [
    ("Behavior correct, reward prediction zero", behavior, zero_prediction),
    ("Behavior set to one half, public rewards supplied", wrong_probability, oracle),
    ("Behavior correct, public rewards supplied", behavior, oracle),
    ("Behavior set to one half, reward prediction zero", wrong_probability, zero_prediction),
]:
    result = evaluate_policy(actions[diagnostic_rows], rewards[diagnostic_rows], supplied_behavior[diagnostic_rows], smooth[diagnostic_rows], supplied_predictions[diagnostic_rows])
    truth = full_information_value(frame, smooth, diagnostic_rows)
    nuisance_rows.append({"Diagnostic case": name, "IPW value": result["ipw_value"], "DR value": result["dr_value"], "Full-information value": truth, "DR absolute error": abs(result["dr_value"] - truth)})
nuisance_scenarios = pd.DataFrame(nuisance_rows)
save_table(nuisance_scenarios, "dr_nuisance_scenarios", "Algebraic nuisance-input diagnostics for the smooth momentum policy.", "tab:cim9-dr-nuisance-scenarios", fit=True)

invalid_checks = pd.DataFrame([
    capture_failure("Prediction matrix with one row missing", lambda: evaluate_policy(actions, rewards, behavior, candidates[0], predictions[:-1])),
    capture_failure("Action outside the declared action set", lambda: evaluate_policy(np.where(np.arange(len(actions)) == 0, 2, actions), rewards, behavior, candidates[0], predictions)),
    capture_failure("Target probability with incompatible shape", lambda: evaluate_policy(actions, rewards, behavior, candidates[0, :, :1], predictions)),
])
save_table(invalid_checks, "dr_invalid_input_checks", "Required rejections for incompatible policy-evaluation inputs.", "tab:cim9-dr-invalid-input-checks", fit=True)

section_summary = pd.DataFrame([
    {"Result": "Released exact check", "Value": "Known value recovered when either nuisance input is correct"},
    {"Result": "Reward predictions", "Value": "Every evaluated date is predicted from earlier logged dates"},
    {"Result": "Public benchmark", "Value": "Both action rewards permit direct full-information comparison"},
    {"Result": "Oracle diagnostic", "Value": "Algebraic check only and not a deployable prediction method"},
    {"Result": "Final-period role", "Value": "Evaluation only after policy selection"},
])
save_table(section_summary, "dr_section_summary", "Summary of the doubly robust evaluation.", "tab:cim9-dr-section-summary", fit=True)

comparison.to_csv(FIGURE_DATA / "dr_policy_values.csv", index=False)
figure, axis = plt.subplots(figsize=(9.2, 5.6), constrained_layout=True)
positions = np.arange(len(comparison))
axis.bar(positions - 0.24, comparison["Full-information value"], 0.24, color=PALETTE["green"], label="Full information")
axis.bar(positions, comparison["IPW value"], 0.24, color=PALETTE["cyan"], label="IPW")
axis.bar(positions + 0.24, comparison["DR value"], 0.24, color=PALETTE["amber"], label="Doubly robust")
axis.set_xticks(positions, ["Defensive", "Cyclical", "Threshold", "Smooth", "Volatility\naware"])
axis.set_ylabel("Mean daily return")
axis.set_title("Held-Out Policy Values")
axis.legend()
finish_axis(axis)
save_figure(figure, "dr_policy_values")

full_scores = np.sum(smooth[heldout] * potential[heldout], axis=1)
running = pd.DataFrame({
    "Date": frame.index[heldout],
    "Full information": np.cumsum(full_scores) / np.arange(1, len(heldout) + 1),
    "IPW": np.cumsum(smooth_result["ipw_scores"]) / np.arange(1, len(heldout) + 1),
    "Doubly robust": np.cumsum(smooth_result["dr_scores"]) / np.arange(1, len(heldout) + 1),
})
running.to_csv(FIGURE_DATA / "dr_running_values.csv", index=False)
figure, axis = plt.subplots(figsize=(9.0, 5.4), constrained_layout=True)
axis.plot(running["Date"], running["Full information"], color=PALETTE["green"], linewidth=2, label="Full information")
axis.plot(running["Date"], running["IPW"], color=PALETTE["cyan"], linewidth=1.5, label="IPW")
axis.plot(running["Date"], running["Doubly robust"], color=PALETTE["amber"], linewidth=1.5, label="Doubly robust")
axis.set_ylabel("Running mean daily return")
axis.set_title("Running Held-Out Values for Smooth Momentum")
axis.legend()
finish_axis(axis)
save_figure(figure, "dr_running_values")

,Policy,Full-information value,IPW value,DR value,IPW absolute error,DR absolute error,Effective sample size
0,Always defensive,0.000134,0.000426,0.000336,0.000292,0.000202,149.428904
1,Always cyclical,0.00123,0.002199,0.002187,0.00097,0.000958,167.880875
2,Positive spread threshold,0.000955,0.001157,0.001189,0.000202,0.000234,224.079433
3,Smooth momentum,0.000967,0.001278,0.001275,0.000311,0.000308,322.098652
4,Volatility aware rule,0.000915,0.001012,0.000994,9.65955e-05,7.85684e-05,345.418543


,Score,Mean,Standard deviation,Minimum,Median,Maximum
0,IPW,0.001278,0.016163,-0.06958,0.000638,0.099174
1,Doubly robust,0.001275,0.016277,-0.066934,0.001288,0.103981


,Diagnostic case,IPW value,DR value,Full-information value,DR absolute error
0,"Behavior correct, reward prediction zero",0.00047,0.00047,0.000913,0.000443
1,"Behavior set to one half, public rewards supplied",0.000883,0.000913,0.000913,0
2,"Behavior correct, public rewards supplied",0.00047,0.000913,0.000913,0
3,"Behavior set to one half, reward prediction zero",0.000883,0.000883,0.000913,3.04007e-05


,Check,Rejected,Explanation
0,Prediction matrix with one row missing,Pass,Stopped as required
1,Action outside the declared action set,Pass,Stopped as required
2,Target probability with incompatible shape,Pass,Stopped as required


,Result,Value
0,Released exact check,Known value recovered when either nuisance inp...
1,Reward predictions,Every evaluated date is predicted from earlier...
2,Public benchmark,Both action rewards permit direct full-informa...
3,Oracle diagnostic,Algebraic check only and not a deployable pred...
4,Final-period role,Evaluation only after policy selection


<Figure size 920x560 with 1 Axes>

<Figure size 900x540 with 1 Axes>